**Batch preprocessing** <br>
This notebook runs the preprocessing 

[] Add the behavioral metadata (that is in BIDS format)
[] Does it also work with only encoding epochs / What to do with participants where enc =/= 64, or retrieval =/= 84?
[] 

**Cell 1: Imports and Environment Setup**

In [14]:
# ==========================================
# 0. IMPORTS & GLOBAL PATHS
# ==========================================
import mne
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
from autoreject import AutoReject
import matplotlib
matplotlib.use('Agg')  # No interactive popups during batch runs
import matplotlib.pyplot as plt
mne.viz.set_browser_backend("matplotlib")

'matplotlib'

**Cell 2: Path Definitions & Global Variables**

In [ ]:
# ==========================================
# 2. Path Definitions & Global Variables 
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"
config_path = derivatives_dir / "preprocessing_config.csv"

In [17]:
# ==========================================
# 1. HELPER: SAVE A FIGURE SAFELY
# ==========================================
def save_fig(fig, out_dir, fname):
    """Save and close a figure, creating the directory if needed."""
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / fname, dpi=100, bbox_inches='tight')
    plt.close(fig)


**Load Configuration**

In [ ]:
config_df = pd.read_csv(config_path)
# Convert string representations of lists back to actual Python lists if necessary
# display(config_df.head())


**Define the Subject-Level Processing Function**

I need to add the metadata merging

In [ ]:
# ==========================================
# 2. MAIN PREPROCESSING FUNCTION
# ==========================================
def preprocess_subject(
    subject_id,
    bad_channels,
    bad_ics,
    l_freq=0.1,
    h_freq=40.0,
    notch_freq=50.0,
    resample_sfreq=250.0,
    epoch_tmin=-0.5,
    epoch_tmax=1.0,
    baseline=(None, 0),
    event_map={'enc_fixation': 110, 'ret_fixation': 210},
    overwrite=False,
):
    """
    Run the full preprocessing pipeline for one subject.

    Parameters
    ----------
    subject_id : str
        BIDS subject label, e.g. "23".
    bad_channels : list of str
        Channels to mark as bad and interpolate, from notebook 2's QC.
    bad_ics : list of int
        ICA component indices to exclude, from notebook 2's QC.
    overwrite : bool
        If False and output files already exist, skip this subject
        (idempotent re-runs — safe to re-execute the batch loop).

    Returns
    -------
    dict
        Summary log (also saved as JSON) with key metrics for this subject.
    """
    subj_deriv_dir = derivatives_dir / f"sub-{subject_id}" / "eeg"
    plots_dir = subj_deriv_dir / "qc_plots"
    log_path = subj_deriv_dir / f"sub-{subject_id}_preproc_log.json"
    beh_path = bids_root / f"sub-{subject_id}" / "beh" / f"sub-{subject_id}_task-loc_label-merged_beh.csv"
    epochs_enc_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-encoding_epo.fif"
    epochs_ret_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-retrieval_epo.fif"

    # --- Idempotency check: skip if already done ---
    if not overwrite and epochs_enc_path.exists() and epochs_ret_path.exists():
        print(f"sub-{subject_id}: outputs already exist, skipping (overwrite=False).")
        with open(log_path, 'r') as f:
            return json.load(f)

    log = {"subject": subject_id, "steps_completed": [], "warnings": []}

    # ==========================================
    # STEP 1: LOAD RAW + APPLY BAD CHANNELS
    # ==========================================
    bids_path = BIDSPath(subject=subject_id, task='loc', datatype='eeg', root=bids_root)
    raw = read_raw_bids(bids_path=bids_path, verbose='error')
    raw.load_data()
    raw.info['bads'] = bad_channels
    log["bad_channels"] = bad_channels
    log["steps_completed"].append("load_raw")

    # Butterfly plot BEFORE filtering
    fig_before = raw.copy().pick('eeg').plot(
        duration=10, n_channels=len(raw.ch_names), show=False,
        scalings=dict(eeg=20e-6), butterfly=True,
        theme='light'  # unrelated, but explicit is nice
    )
    save_fig(fig_before, plots_dir, "01_butterfly_before_filter.png")

    # ==========================================
    # STEP 2: FILTERING (low, high, notch)
    # ==========================================
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    raw.notch_filter(notch_freq, verbose=False)
    log["filter_params"] = {"l_freq": l_freq, "h_freq": h_freq, "notch_freq": notch_freq}
    log["steps_completed"].append("filter")

    # Butterfly plot AFTER filtering
    fig_after = raw.copy().pick('eeg').plot(
        duration=10, n_channels=len(raw.ch_names), show=False,
        scalings=dict(eeg=20e-6), butterfly=True
    )
    save_fig(fig_after, plots_dir, "02_butterfly_after_filter.png")

    # Interpolate bad channels now (before ICA apply / epoching)
    if bad_channels:
        raw.interpolate_bads(reset_bads=True, verbose=False)
        log["steps_completed"].append("interpolate_bads")

    # ==========================================
    # STEP 3: DOWNSAMPLE
    # ==========================================
    raw.resample(resample_sfreq, verbose=False)
    log["resample_sfreq"] = resample_sfreq
    log["steps_completed"].append("resample")

    # ==========================================
    # STEP 4: APPLY ICA (using bad_ics from notebook 2)
    # ==========================================
    # NOTE: this assumes you saved the fitted ICA object itself from notebook 2
    # (not just the component indices) — ICA.apply() needs the actual unmixing
    # matrix, which is subject-specific and can't be recomputed from indices alone.
    ica_path = derivatives_dir / f"sub-{subject_id}" / "eeg" / "ica_qc" / f"sub-{subject_id}_ica.fif"
    if not ica_path.exists():
        raise FileNotFoundError(
            f"No saved ICA solution found for sub-{subject_id} at {ica_path}. "
            f"Save ica.save(...) at the end of notebook 2."
        )
    ica = mne.preprocessing.read_ica(ica_path)
    ica.exclude = bad_ics
    ica.apply(raw)
    log["excluded_ics"] = bad_ics
    log["steps_completed"].append("ica_apply")

    # ==========================================
    # STEP 5: EPOCHING (encoding + retrieval, separately)
    # ==========================================
   
   # Extract events from annotations
    events, _ = mne.events_from_annotations(raw, verbose=False)

    target_event_id = {
    'Target/SC/PC': event_id['tgt_sc_pc'], # Zieht sich die von MNE vergebene ID (48)
    'Target/SC/PI': event_id['tgt_sc_pi'], # Zieht sich die ID 49
    'Target/SI/PC': event_id['tgt_si_pc'], # Zieht sich die ID 50
    'Target/SI/PI': event_id['tgt_si_pi']  # Zieht sich die ID 51
}
    epochs_dict = {}
    for phase, ev_id in [('encoding', event_map.get('enc_fixation')),
                          ('retrieval', event_map.get('ret_fixation'))]:
        if ev_id is None or ev_id not in events[:, 2]:
            log["warnings"].append(f"No events found for {phase} phase — skipped.")
            continue
        epochs = mne.Epochs(
            raw, events, event_id={f"{phase}": ev_id},
            tmin=epoch_tmin, tmax=epoch_tmax, baseline=None,
            preload=True, reject_by_annotation=True, verbose=False
        )
        epochs_dict[phase] = epochs

    log["steps_completed"].append("epoching")
    log["n_epochs_pre_autoreject"] = {k: len(v) for k, v in epochs_dict.items()}

    # ==========================================
    # STEP 6: AUTOREJECT (separately per phase)
    # ==========================================
    epochs_clean = {}
    for phase, epochs in epochs_dict.items():
        # Evoked BEFORE autoreject
        fig_evoked_before = epochs.average().plot(show=False)
        save_fig(fig_evoked_before, plots_dir, f"03_evoked_{phase}_before_autoreject.png")

        ar = AutoReject(random_state=97, n_jobs=1, verbose=False)
        epochs_ar, reject_log = ar.fit_transform(epochs, return_log=True)
        epochs_clean[phase] = epochs_ar

        log.setdefault("n_epochs_post_autoreject", {})[phase] = len(epochs_ar)
        log.setdefault("autoreject_pct_dropped", {})[phase] = round(
            100 * (1 - len(epochs_ar) / len(epochs)), 1
        )

        # Evoked AFTER autoreject
        fig_evoked_after = epochs_ar.average().plot(show=False)
        save_fig(fig_evoked_after, plots_dir, f"04_evoked_{phase}_after_autoreject.png")

        # Reject log visualization (which epochs/channels were dropped/interpolated)
        fig_reject = reject_log.plot(show=False)
        save_fig(fig_reject, plots_dir, f"05_autoreject_log_{phase}.png")

    log["steps_completed"].append("autoreject")

    # ==========================================
    # STEP 7: BASELINE CORRECTION
    # ==========================================
    for phase, epochs in epochs_clean.items():
        epochs.apply_baseline(baseline)
    log["steps_completed"].append("baseline_correction")

    # ==========================================
    # STEP 8: AVERAGE RE-REFERENCING
    # ==========================================
    for phase, epochs in epochs_clean.items():
        epochs.set_eeg_reference('average', projection=False, verbose=False)
    log["steps_completed"].append("average_reference")

    # ==========================================
    # STEP 9: SAVE OUTPUTS
    # ==========================================
    subj_deriv_dir.mkdir(parents=True, exist_ok=True)
    epochs_clean['encoding'].save(epochs_enc_path, overwrite=overwrite) if 'encoding' in epochs_clean else None
    epochs_clean['retrieval'].save(epochs_ret_path, overwrite=overwrite) if 'retrieval' in epochs_clean else None

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"sub-{subject_id}: preprocessing complete. "
          f"Encoding epochs: {log.get('n_epochs_post_autoreject', {}).get('encoding', 'N/A')}, "
          f"Retrieval epochs: {log.get('n_epochs_post_autoreject', {}).get('retrieval', 'N/A')}")

    return log



**Option 1: Test Pipeline on A Single Subject** - it works! :)

In [20]:
# ==========================================
# 3. SINGLE-SUBJECT TEST RUN
# ==========================================
# Test on one subject before running the full batch
test_log = preprocess_subject(
    subject_id="03",
    bad_channels=[],
    bad_ics=["T7", "T8"],
    overwrite=True,
)

FileNotFoundError: No saved ICA solution found for sub-03 at c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-03\eeg\ica_qc\sub-03_ica.fif. Save ica.save(...) at the end of notebook 2.

**Option 2: Run Pipeline for all subjects**

In [ ]:
# ==========================================
# 4. BATCH RUN OVER ALL MANUALLY-CHECKED SUBJECTS
# ==========================================
df_config = pd.read_csv(config_path, dtype={'subject': str})

# Only process subjects that are:
# - not behaviorally excluded
# - have complete EEG recordings for both phases (adjust if partial data is acceptable)
ready_mask = (
    (~df_config['is_excluded'].astype(bool)) &
    (df_config['eeg_enc_recorded'] == True) &
    (df_config['eeg_ret_recorded'] == True)
)
subjects_to_process = df_config[ready_mask]

print(f"{len(subjects_to_process)} of {len(df_config)} subjects ready for batch preprocessing.")

batch_summary = []
for _, row in subjects_to_process.iterrows():
    subj_id = row['subject']
    bad_chs = [c.strip() for c in row['bad_channels'].split(',')] if pd.notna(row['bad_channels']) and row['bad_channels'] else []
    bad_ic_list = [int(x.strip()) for x in row['manual_bad_icas'].split(',')] if pd.notna(row['manual_bad_icas']) and row['manual_bad_icas'] else []

    try:
        log = preprocess_subject(
            subject_id=subj_id,
            bad_channels=bad_chs,
            bad_ics=bad_ic_list,
            overwrite=False,  # skip subjects already processed
        )
        batch_summary.append({"subject": subj_id, "status": "success", **log.get("n_epochs_post_autoreject", {})})
    except Exception as e:
        print(f"sub-{subj_id}: FAILED — {e}")
        batch_summary.append({"subject": subj_id, "status": "failed", "error": str(e)})

df_batch_summary = pd.DataFrame(batch_summary)
df_batch_summary.to_csv(derivatives_dir / "batch_preprocessing_summary.csv", index=False)
display(df_batch_summary)